# 🧠 Brain Tumor MRI — DINOv2-B/14 | RESUME từ Phase1 Epoch 7

**Notebook này resume từ checkpoint `phase1_epoch_07.pt`**
- ✅ Train nốt Phase 1 epoch 8 (epoch cuối)
- ✅ Chạy toàn bộ Phase 2 (12 epochs, unfreeze 4 blocks cuối)
- ✅ Đánh giá và lưu kết quả

> **Yêu cầu input:** Notebook dataset `DINOv2_ViT_B14z_canceled` (chứa checkpoints từ lần chạy trước)

In [ ]:
!pip install "numpy<2.0" --force-reinstall -q
!pip install pandas scikit-learn --force-reinstall --no-cache-dir -q

In [ ]:
import os, json, glob, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

import timm
import kagglehub
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

In [ ]:
# ============================================================
# CẤU HÌNH 
# ============================================================
RANDOM_SEED = 32

def seed_everything(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_multi_gpu = torch.cuda.is_available() and torch.cuda.device_count() > 1
print(f'Device: {device} | GPU count: {torch.cuda.device_count()} | DataParallel: {use_multi_gpu}')

IMG_SIZE   = 518
BATCH_SIZE = 16
NUM_CLASSES = 30
VAL_RATIO  = 0.15
TEST_RATIO = 0.15

# Phase 1: chỉ train NỐT 1 epoch còn lại (epoch 8)
PHASE1_EPOCHS_TOTAL = 8      # tổng epochs phase 1
PHASE1_DONE_EPOCHS  = 7      # đã train xong bao nhiêu epoch
PHASE1_LR           = 1e-3

# Phase 2: unfreeze 4 blocks cuối, 12 epochs
PHASE2_EPOCHS = 12
PHASE2_LR     = 5e-5
PHASE2_LAST_N_BLOCKS = 4

FOCAL_GAMMA  = 2.0
FOCAL_ALPHA  = 0.25
WEIGHT_DECAY = 1e-4
NUM_WORKERS  = 4

# ── Đường dẫn output ──────────────────────────────────────
OUTPUT_DIR = Path('./dino_v2_m_outputs')
CKPT_DIR   = OUTPUT_DIR / 'checkpoints'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

BEST_WEIGHT = OUTPUT_DIR / 'best_weights.pt'
LOG_CSV     = OUTPUT_DIR / 'training_log.csv'

# ── Checkpoint resume ─────────────────────────────────────
# Tìm checkpoint phase1_epoch_07 trong Kaggle input
RESUME_CKPT_CANDIDATES = [
    Path('/kaggle/input/dinoV2_ViT_B14z_canceled/dino_v2_m_outputs/checkpoints/phase1_epoch_07.pt'),
    Path('/kaggle/input/dinov2-vit-b14z-canceled/dino_v2_m_outputs/checkpoints/phase1_epoch_07.pt'),
]
# Thêm wildcard search phòng trường hợp tên dataset khác
import glob as _glob
_found = _glob.glob('/kaggle/input/**/phase1_epoch_07.pt', recursive=True)
RESUME_CKPT_CANDIDATES += [Path(p) for p in _found]

RESUME_CKPT = None
for candidate in RESUME_CKPT_CANDIDATES:
    if candidate.exists():
        RESUME_CKPT = candidate
        break

if RESUME_CKPT is None:
    raise FileNotFoundError(
        '❌ Không tìm thấy phase1_epoch_07.pt!\n'
        'Hãy thêm notebook dataset "DINOv2_ViT_B14z_final" vào Input của notebook này.'
    )
print(f'✅ Tìm thấy checkpoint resume: {RESUME_CKPT}')

# Best weights cũ (để so sánh tiếp)
OLD_BEST_WEIGHT_CANDIDATES = [
    RESUME_CKPT.parent.parent / 'best_weights.pt',
]
_found_best = _glob.glob('/kaggle/input/**/best_weights.pt', recursive=True)
OLD_BEST_WEIGHT_CANDIDATES += [Path(p) for p in _found_best]

OLD_BEST_WEIGHT = None
for candidate in OLD_BEST_WEIGHT_CANDIDATES:
    if candidate.exists():
        OLD_BEST_WEIGHT = candidate
        break
print(f'Best weight cũ: {OLD_BEST_WEIGHT}')

In [ ]:
# ============================================================
# TẢI VÀ PHÂN CHIA DATASET — giữ NGUYÊN seed như notebook gốc
# ============================================================
DATASET_PATH = Path(kagglehub.dataset_download('fernando2rad/brain-tumor-mri-images-30-classes'))
print(f'Dataset path: {DATASET_PATH}')

all_files = []
for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
    all_files.extend(glob.glob(str(DATASET_PATH / '**' / ext), recursive=True))

records = []
for fp in all_files:
    class_name = Path(fp).parts[-2]
    records.append({'filepath': fp, 'class_name': class_name})

df = pd.DataFrame(records)
CLASS_NAMES = sorted(df['class_name'].unique().tolist())
class2idx = {c: i for i, c in enumerate(CLASS_NAMES)}
df['label'] = df['class_name'].map(class2idx)

print(f'Tổng số file ảnh: {len(df):,} | Số class: {len(CLASS_NAMES)}')

# Split — PHẢI giữ nguyên random_state=32 để đồng nhất với notebook gốc
train_list, temp_list = [], []
for cls in CLASS_NAMES:
    cls_df = df[df['class_name'] == cls]
    tr, tmp = train_test_split(cls_df, test_size=(VAL_RATIO + TEST_RATIO), random_state=RANDOM_SEED, shuffle=True)
    train_list.append(tr)
    temp_list.append(tmp)

train_df = pd.concat(train_list).reset_index(drop=True)
temp_df  = pd.concat(temp_list).reset_index(drop=True)

val_list, test_list = [], []
for cls in CLASS_NAMES:
    cls_df = temp_df[temp_df['class_name'] == cls]
    v, t = train_test_split(cls_df, test_size=0.5, random_state=RANDOM_SEED, shuffle=True)
    val_list.append(v)
    test_list.append(t)

val_df  = pd.concat(val_list).reset_index(drop=True)
test_df = pd.concat(test_list).reset_index(drop=True)

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

In [ ]:
# ============================================================
# DATASET & DATALOADER
# ============================================================
class BrainTumorDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['filepath']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, int(row['label'])

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_loader = DataLoader(BrainTumorDataset(train_df, train_transform), batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(BrainTumorDataset(val_df,   val_transform),   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(BrainTumorDataset(test_df,  val_transform),   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}')

In [ ]:
# ============================================================
# KHỞI TẠO MODEL — HEAD KHỚP VỚI CHECKPOINT CŨ
# ============================================================
# Head có kiến trúc:
#   0: LayerNorm(768)
#   1: Linear(768 -> 512)
#   2: GELU
#   3: Dropout
#   4: Linear(512 -> 256)
#   5: GELU
#   6: Dropout
#   7: Linear(256 -> 30)
class DINOv2Classifier(nn.Module):
    def __init__(self, backbone_name='vit_base_patch14_dinov2', num_classes=30, dropout=0.3):
        super().__init__()
        self.backbone_name = backbone_name
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        feat_dim = self.backbone.num_features   # 768 cho ViT-B/14
        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),      # 0
            nn.Linear(feat_dim, 512),    # 1
            nn.GELU(),                   # 2
            nn.Dropout(dropout),         # 3
            nn.Linear(512, 256),         # 4
            nn.GELU(),                   # 5
            nn.Dropout(dropout),         # 6
            nn.Linear(256, num_classes), # 7
        )

    def forward(self, x):
        feats = self.backbone(x)
        return self.head(feats)

def get_base_model(m):
    return m.module if isinstance(m, nn.DataParallel) else m

# Khởi tạo model
base_model = DINOv2Classifier('vit_base_patch14_dinov2', num_classes=NUM_CLASSES)

if use_multi_gpu:
    model = nn.DataParallel(base_model)
else:
    model = base_model

model = model.to(device)
print(f'✅ Model khởi tạo: {get_base_model(model).backbone_name}')
print('   Head:')
for i, layer in enumerate(get_base_model(model).head):
    print(f'     {i}: {layer}')

In [ ]:
# ============================================================
# LOAD CHECKPOINT PHASE1 EPOCH 7
# ============================================================
checkpoint = torch.load(RESUME_CKPT, map_location=device)
get_base_model(model).load_state_dict(checkpoint['model_state_dict'])
print(f'✅ Đã load checkpoint: {RESUME_CKPT.name}')
print(f'   Phase: {checkpoint["phase"]} | Epoch: {checkpoint["epoch"]} | val_f1: {checkpoint["val_f1"]:.4f}')

# Load best val_f1 cũ để tiếp tục so sánh khi save best_weights
if OLD_BEST_WEIGHT and OLD_BEST_WEIGHT.exists():
    old_best_ckpt = torch.load(OLD_BEST_WEIGHT, map_location='cpu')
    GLOBAL_BEST_VAL_F1 = old_best_ckpt['val_f1']
    print(f'   Best val_f1 từ trước: {GLOBAL_BEST_VAL_F1:.4f} (từ {old_best_ckpt["phase"]} epoch {old_best_ckpt["epoch"]})')
    # Copy best_weights cũ vào output để không mất
    import shutil
    shutil.copy2(OLD_BEST_WEIGHT, BEST_WEIGHT)
    print(f'   Đã copy best_weights cũ → {BEST_WEIGHT}')
else:
    GLOBAL_BEST_VAL_F1 = checkpoint['val_f1']
    print(f'   Không tìm thấy best_weights cũ, dùng val_f1 của epoch 7: {GLOBAL_BEST_VAL_F1:.4f}')

In [ ]:
# ============================================================
# FOCAL LOSS & HELPER FUNCTIONS
# ============================================================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, logits, targets):
        ce = nn.functional.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce)
        loss = self.alpha * (1 - pt) ** self.gamma * ce
        return loss.mean() if self.reduction == 'mean' else loss

criterion = FocalLoss(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA)

def set_trainable_backbone(model, last_n_blocks=None):
    base = get_base_model(model)
    for p in base.backbone.parameters():
        p.requires_grad = False
    if last_n_blocks is None:
        for p in base.backbone.parameters():
            p.requires_grad = True
    elif last_n_blocks > 0:
        if hasattr(base.backbone, 'blocks'):
            for block in base.backbone.blocks[-last_n_blocks:]:
                for p in block.parameters():
                    p.requires_grad = True
        if hasattr(base.backbone, 'norm'):
            for p in base.backbone.norm.parameters():
                p.requires_grad = True
    for p in base.head.parameters():
        p.requires_grad = True

def make_optimizer(lr):
    return torch.optim.AdamW(
        filter(lambda p: p.requires_grad, get_base_model(model).parameters()),
        lr=lr, weight_decay=WEIGHT_DECAY
    )

@torch.no_grad()
def evaluate(loader):
    base = get_base_model(model)
    base.eval()
    running_loss = 0.0
    y_true, y_pred, y_prob = [], [], []
    for images, targets in loader:
        images  = images.to(device)
        targets = targets.to(device)
        logits  = base(images)
        loss    = criterion(logits, targets)
        running_loss += loss.item() * images.size(0)
        probs = torch.softmax(logits, dim=1)
        y_true.append(targets.cpu())
        y_pred.append(probs.argmax(dim=1).cpu())
        y_prob.append(probs.cpu())
    y_true = torch.cat(y_true).numpy()
    y_pred = torch.cat(y_pred).numpy()
    y_prob = torch.cat(y_prob).numpy()
    return (running_loss / len(loader.dataset),
            accuracy_score(y_true, y_pred),
            f1_score(y_true, y_pred, average='macro', zero_division=0),
            y_true, y_pred, y_prob)

print('✅ FocalLoss, helpers khởi tạo xong')

In [ ]:
# ============================================================
# FIT STAGE — có hỗ trợ RESUME (epoch_offset & global best)
# ============================================================
def fit_stage(phase_name, epochs, lr, last_n_blocks, history,
              epoch_offset=0, global_best_val_f1=-1.0):
    """
    epoch_offset: số epoch đã chạy trước đó của phase này
                  (dùng để đặt tên file checkpoint đúng số)
    global_best_val_f1: val_f1 tốt nhất từ toàn bộ quá trình train
                        (gồm cả các epoch trước khi resume)
    """
    base = get_base_model(model)
    set_trainable_backbone(model, last_n_blocks=last_n_blocks)
    optimizer = make_optimizer(lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=2, min_lr=1e-6)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

    best_val_f1 = global_best_val_f1  # kế thừa best từ trước

    for epoch in range(1, epochs + 1):
        global_epoch = epoch_offset + epoch  # epoch tuyệt đối

        base.train()
        train_loss = 0.0
        y_true_t, y_pred_t = [], []

        for images, targets in train_loader:
            images  = images.to(device)
            targets = targets.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type,
                                dtype=torch.float16,
                                enabled=(device.type == 'cuda')):
                logits = model(images)
                loss   = criterion(logits, targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            y_true_t.append(targets.detach().cpu())
            y_pred_t.append(preds.detach().cpu())

        y_true_t   = torch.cat(y_true_t).numpy()
        y_pred_t   = torch.cat(y_pred_t).numpy()
        train_acc  = accuracy_score(y_true_t, y_pred_t)
        train_f1   = f1_score(y_true_t, y_pred_t, average='macro', zero_division=0)
        train_loss /= len(train_loader.dataset)

        val_loss, val_acc, val_f1, *_ = evaluate(val_loader)
        scheduler.step(val_f1)

        history.append({
            'phase': phase_name, 'epoch': global_epoch,
            'train_loss': train_loss, 'train_acc': train_acc,
            'train_f1_macro': train_f1,
            'val_loss': val_loss, 'val_acc': val_acc,
            'val_f1_macro': val_f1,
            'lr': optimizer.param_groups[0]['lr']
        })

        # Lưu checkpoint theo đúng số epoch tuyệt đối
        ckpt_path = CKPT_DIR / f'{phase_name}_epoch_{global_epoch:02d}.pt'
        torch.save({
            'model_state_dict': base.state_dict(),
            'phase': phase_name,
            'epoch': global_epoch,
            'val_f1': val_f1,
            'backbone_name': base.backbone_name
        }, ckpt_path)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save({
                'model_state_dict': base.state_dict(),
                'phase': phase_name,
                'epoch': global_epoch,
                'val_f1': val_f1,
                'backbone_name': base.backbone_name
            }, BEST_WEIGHT)
            print(f'  ✅ Best weights updated: epoch {global_epoch:02d} | val_f1_macro={val_f1:.4f}')

        print(f'[{phase_name}] epoch {global_epoch:02d} | '
              f'train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | '
              f'train_acc={train_acc:.4f} | val_acc={val_acc:.4f} | '
              f'val_f1={val_f1:.4f}')

    return best_val_f1

print('✅ fit_stage() đã sẵn sàng')

In [ ]:
# ============================================================
# TRAIN: Phase 1 epoch 8 → Phase 2 (12 epochs)
# ============================================================
history = []

# ── Phase 1: Train nốt 1 epoch còn lại (epoch 8) ──────────
remaining_phase1 = PHASE1_EPOCHS_TOTAL - PHASE1_DONE_EPOCHS   # = 1
print(f'🚀 Phase 1 — Train nốt {remaining_phase1} epoch còn lại (epoch {PHASE1_DONE_EPOCHS+1}/{PHASE1_EPOCHS_TOTAL})')
current_best = fit_stage(
    phase_name='phase1',
    epochs=remaining_phase1,
    lr=PHASE1_LR,
    last_n_blocks=0,          # backbone frozen
    history=history,
    epoch_offset=PHASE1_DONE_EPOCHS,       # bắt đầu đếm từ epoch 8
    global_best_val_f1=GLOBAL_BEST_VAL_F1 # kế thừa best cũ
)

# ── Phase 2 ────────────────────────────────────────────────
print(f'\n🚀 Phase 2 — Unfreeze {PHASE2_LAST_N_BLOCKS} blocks cuối | {PHASE2_EPOCHS} epochs')
current_best = fit_stage(
    phase_name='phase2',
    epochs=PHASE2_EPOCHS,
    lr=PHASE2_LR,
    last_n_blocks=PHASE2_LAST_N_BLOCKS,
    history=history,
    epoch_offset=0,           # đặt lại từ epoch 1 cho phase2
    global_best_val_f1=current_best
)

# Lưu log
pd.DataFrame(history).to_csv(LOG_CSV, index=False)
print(f'\n✅ Đã lưu log: {LOG_CSV}')
print(f'✅ Best val_f1 toàn quá trình: {current_best:.4f}')

In [ ]:
# ============================================================
# LEARNING CURVES
# ============================================================
log_df = pd.read_csv(LOG_CSV)

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
for ax, (y1, y2, title) in zip(
    axes.flat,
    [('train_loss','val_loss','Loss'),
     ('train_acc','val_acc','Accuracy'),
     ('train_f1_macro','val_f1_macro','F1 Macro'),
     (None, None, 'Learning Rate')]
):
    if y1 is None:
        ax.semilogy(log_df['epoch'], log_df['lr'])
    else:
        ax.plot(log_df['epoch'], log_df[y1], label=f'Train {y1.split("_")[1].capitalize()}')
        ax.plot(log_df['epoch'], log_df[y2], label=f'Val {y2.split("_")[1].capitalize()}')
        ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_title(title)
    ax.set_xlabel('Epoch')

# Vẽ đường kẻ phân chia Phase 1 / Phase 2
phase1_last = log_df[log_df['phase']=='phase1']['epoch'].max()
for ax in axes.flat:
    ax.axvline(x=phase1_last + 0.5, color='red', linestyle='--', alpha=0.5, label='Phase1→2')

axes.flat[0].legend()
plt.suptitle('Learning Curves — DINOv2-B/14 (Resume from epoch 7)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu learning_curves.png')

In [ ]:
import os, glob

# File nằm trong /kaggle/input/ (mount từ notebook input)
found = glob.glob("/kaggle/input/**/*.pt", recursive=True)

print("=== File .pt trong /kaggle/input/ ===")
for f in found:
    print(f"  {f}  ({os.path.getsize(f)/1e6:.1f} MB)")

In [ ]:
# ── Load best weights ──────────────────────────────────────────────────
CKPT_PATH = "/kaggle/input/notebooks/pouncer098/dinov2-vit-b14z-final/dino_v2_m_outputs/best_weights.pt"
assert os.path.exists(CKPT_PATH), f"❌ Không tìm thấy: {CKPT_PATH}"

checkpoint = torch.load(CKPT_PATH, map_location=device)
get_base_model(model).load_state_dict(checkpoint["model_state_dict"])
get_base_model(model).eval()

print(f"✅ Loaded — phase: {checkpoint['phase']} | "
      f"epoch: {checkpoint['epoch']} | "
      f"val_f1: {checkpoint['val_f1']:.4f}")

# ── Đánh giá trên test set ─────────────────────────────────────────────
test_loss, test_acc, test_f1_macro, y_true, y_pred, y_prob = evaluate(test_loader)
f1_weighted = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print(f"\nTest Results:")
print(f"  Accuracy   : {test_acc:.4f}")
print(f"  F1 Macro   : {test_f1_macro:.4f}")
print(f"  F1 Weighted: {f1_weighted:.4f}")
checkpoint = torch.load(CKPT_PATH, map_location=device)

state_dict = checkpoint["model_state_dict"]

# DataParallel wrap → load vào .module bên trong
if isinstance(model, torch.nn.DataParallel):
    model.module.load_state_dict(state_dict)
    print("✅ Loaded vào model.module (DataParallel detected)")
else:
    model.load_state_dict(state_dict)
    print("✅ Loaded vào model")

print(f"   epoch: {checkpoint.get('epoch', '?')}")
print(f"   val_acc: {checkpoint.get('val_acc', '?')}")

model.eval()
print("🔒 model.eval() — sẵn sàng test")

In [ ]:
# ============================================================
# CLASSIFICATION REPORT
# ============================================================
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4, zero_division=0)
print(report)
with open(OUTPUT_DIR / 'classification_report.txt', 'w', encoding='utf-8') as f:
    f.write(report)
print('✅ Đã lưu classification_report.txt')

In [ ]:
# ============================================================
# CONFUSION MATRIX
# ============================================================
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(22, 18))
sns.heatmap(cm, annot=False, cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.3, ax=ax)
ax.set_title(f'Confusion Matrix — DINOv2-B/14\nAccuracy: {test_acc:.4f} | F1 Macro: {test_f1_macro:.4f}')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu confusion_matrix.png')

In [ ]:
# ============================================================
# VISUALIZE DỰ ĐOÁN ĐÚNG / SAI
# ============================================================
def show_predictions(df_subset, y_true_arr, y_pred_arr, y_probs_arr, title, n=12, correct=True):
    idxs = np.where(y_true_arr == y_pred_arr)[0] if correct else np.where(y_true_arr != y_pred_arr)[0]
    if len(idxs) == 0:
        print(f'Không có ảnh nào ({title})')
        return
    chosen = np.random.choice(idxs, size=min(n, len(idxs)), replace=False)
    ncols = 4
    nrows = (len(chosen) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 4))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis('off')
    for ax, i in zip(axes, chosen):
        img   = Image.open(df_subset.iloc[i]['filepath']).convert('RGB')
        true_lbl = CLASS_NAMES[y_true_arr[i]]
        pred_lbl = CLASS_NAMES[y_pred_arr[i]]
        conf     = y_probs_arr[i][y_pred_arr[i]]
        color    = '#27ae60' if correct else '#e74c3c'
        ax.imshow(img)
        ax.set_title(f'True : {true_lbl}\nPred : {pred_lbl}\nConf : {conf:.2%}', fontsize=7.5, color=color)
        ax.axis('off')
    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_predictions(test_df, y_true, y_pred, y_prob, '✅ Dự đoán ĐÚNG', n=12, correct=True)
show_predictions(test_df, y_true, y_pred, y_prob, '❌ Dự đoán SAI',  n=12, correct=False)

In [ ]:
# ============================================================
# LƯU MODEL SUMMARY
# ============================================================
summary = {
    'version': 'v2-resumed',
    'model_family': 'DINOv2',
    'backbone_name': get_base_model(model).backbone_name,
    'img_size': IMG_SIZE,
    'num_classes': NUM_CLASSES,
    'class_names': CLASS_NAMES,
    'random_seed': RANDOM_SEED,
    'resumed_from': str(RESUME_CKPT),
    'phase1_epochs_total': PHASE1_EPOCHS_TOTAL,
    'phase2_epochs': PHASE2_EPOCHS,
    'test_loss': float(test_loss),
    'test_accuracy': float(test_acc),
    'test_f1_macro': float(test_f1_macro),
    'test_f1_weighted': float(f1_weighted),
    'best_weight_path': str(BEST_WEIGHT),
}
with open(OUTPUT_DIR / 'model_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print('✅ Đã lưu model_summary.json')
print('\n=== HOÀN THÀNH ===')
print(f'  Test Accuracy  : {test_acc:.4f}')
print(f'  Test F1 Macro  : {test_f1_macro:.4f}')
print(f'  Test F1 Weighted: {f1_weighted:.4f}')

In [ ]:
import shutil, os
from IPython.display import FileLink, display

SRC = CKPT_PATH  # path đã xác nhận ở trên
DST = "/kaggle/working/dino_v2_m_outputs/best_weights.pt"

# Copy sang working/ (input/ không download trực tiếp được)
shutil.copy2(SRC, DST)
print(f"✅ Copied: {os.path.getsize(DST)/1e6:.1f} MB")

# Tạo link bấm để tải về máy
display(FileLink(DST, result_html_prefix="⬇️ Download: "))